# Short-term memory with `InMemorySaver`

**Short-term memory** = what the bot remembers *within one conversation* (one `thread_id`).
The checkpointer saves the message list after every step, and the next turn on the same
thread starts from it.

`InMemorySaver` keeps it in RAM: perfect for learning and tests, gone when the program stops.

In [1]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

model = ChatGroq(model='openai/gpt-oss-20b', reasoning_effort='low')

In [2]:
# MessagesState is a built-in state: {'messages': Annotated[list, add_messages]}
def chat_node(state: MessagesState):
    return {'messages': [model.invoke(state['messages'])]}


graph = StateGraph(MessagesState)
graph.add_node('chat_node', chat_node)
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

checkpointer = InMemorySaver()
chatbot = graph.compile(checkpointer=checkpointer)

In [3]:
def chat(text, thread_id):
    config = {'configurable': {'thread_id': thread_id}}
    result = chatbot.invoke({'messages': [HumanMessage(content=text)]}, config=config)
    print(f'[{thread_id}] You: {text}\n[{thread_id}] AI:  {result["messages"][-1].content}\n')


chat('Hi! My name is Priyansh and I live in Jaipur.', 'user-1')
chat('I am learning LangGraph.', 'user-1')
chat('What is my name, where do I live and what am I learning?', 'user-1')

[user-1] You: Hi! My name is Priyansh and I live in Jaipur.
[user-1] AI:  Hey Priyansh! 👋  
Nice to meet you—Jaipur is a fantastic city with so much history and culture. How can I help you today? Whether it’s travel tips, local food recommendations, or just a chat, I’m here!



[user-1] You: I am learning LangGraph.
[user-1] AI:  That’s awesome! LangGraph is a powerful framework for building complex, state‑ful language‑model applications. Where are you at right now—just getting started with the basics, or diving into more advanced patterns like multi‑step reasoning or custom state management? Let me know what you’re trying to build or any specific questions you have, and we can walk through examples, best practices, or even troubleshoot code together!



[user-1] You: What is my name, where do I live and what am I learning?
[user-1] AI:  You’re Priyansh, you live in Jaipur, and you’re learning LangGraph.



In [4]:
# Memory is per thread: a new thread_id is a new conversation with an empty memory
chat('What is my name?', 'user-2')

[user-2] You: What is my name?
[user-2] AI:  I don’t have that information. Could you tell me your name?



In [5]:
# What is actually stored for each thread
for thread_id in ['user-1', 'user-2']:
    messages = chatbot.get_state({'configurable': {'thread_id': thread_id}}).values['messages']
    print(f'{thread_id}: {len(messages)} messages ->', [m.type for m in messages])

user-1: 6 messages -> ['human', 'ai', 'human', 'ai', 'human', 'ai']
user-2: 2 messages -> ['human', 'ai']


In [6]:
# The catch: the WHOLE history is sent to the LLM on every turn, so each turn costs more tokens
for i in range(3):
    chat(f'Tell me one short fact about the number {i + 1}.', 'user-1')

messages = chatbot.get_state({'configurable': {'thread_id': 'user-1'}}).values['messages']
for m in messages:
    if m.type == 'ai':
        print('prompt tokens sent on that turn:', m.usage_metadata['input_tokens'])

[user-1] You: Tell me one short fact about the number 1.
[user-1] AI:  1 is the only positive integer that is neither prime nor composite.



[user-1] You: Tell me one short fact about the number 2.
[user-1] AI:  2 is the smallest and the only even prime number.



[user-1] You: Tell me one short fact about the number 3.
[user-1] AI:  3 is the first odd prime number.

prompt tokens sent on that turn: 85
prompt tokens sent on that turn: 153
prompt tokens sent on that turn: 262
prompt tokens sent on that turn: 301
prompt tokens sent on that turn: 335
prompt tokens sent on that turn: 367
